In [1]:
import pandas as pd
from  transformers import T5Tokenizer, Trainer, TrainingArguments, T5ForConditionalGeneration

In [2]:
train_data = pd.read_csv("samsum-train.csv")
val_data = pd.read_csv("samsum-validation.csv")

In [3]:
train_data.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [4]:
train_data["dialogue"][0]

"Amanda: I baked  cookies. Do you want some?\r\nJerry: Sure!\r\nAmanda: I'll bring you tomorrow :-)"

In [5]:
train_data.shape

(14732, 3)

In [6]:
val_data.shape

(818, 3)

In [7]:
# random sampling
train_data = train_data.sample(n=4000, random_state=42).reset_index(drop=True)
train_data = train_data.sample(n=4000, random_state=42).reset_index(drop=True)

In [8]:
train_data.shape

(4000, 3)

## Data pre-processing

In [9]:
import re

def clean_data(text):
    text = re.sub(r"\r\n", " ", text)  # lines
    text = re.sub(r"\s+", " ", text) # spaces
    text = text.strip().lower()
    return text

In [10]:
train_data["dialogue"] = train_data["dialogue"].apply(clean_data)
train_data["summary"] = train_data["summary"].apply(clean_data)

val_data["dialogue"] = val_data["dialogue"].apply(clean_data)
val_data["summary"] = val_data["summary"].apply(clean_data)

## Tokenize

In [11]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

In [12]:
# raw data => tokenized inputs for fine-tuning

def tokenize(data):
    inputs = tokenizer(data['dialogue'], padding="max_length", max_length=512, truncation=True)
    targets = tokenizer(data["summary"], padding='max_length', max_length=512, truncation=True)

    inputs["labels"] = targets['input_ids']  #token ids => add to input as labels
    return inputs

In [13]:
train_dataset = train_data.apply(tokenize, axis=1).tolist()
val_dataset = val_data.apply(tokenize, axis=1).tolist()

In [14]:
train_dataset[0]

{'input_ids': [836, 6961, 10, 3, 13133, 1240, 226, 6, 103, 25, 341, 3884, 22269, 58, 1240, 226, 10, 7102, 3158, 307, 97, 150, 1616, 3, 10, 61, 17945, 6, 3, 23, 103, 6, 125, 31, 7, 95, 58, 836, 6961, 10, 320, 6, 82, 7930, 19, 479, 21, 841, 113, 228, 3884, 376, 6, 78, 3, 23, 816, 81, 25, 5, 1240, 226, 10, 417, 589, 55, 125, 593, 19, 3, 88, 44, 230, 58, 836, 6961, 10, 3, 107, 51, 233, 614, 12, 817, 6, 43, 29, 31, 17, 1380, 3, 117, 61, 3, 23, 317, 3, 88, 131, 2746, 12, 36, 6720, 295, 6, 78, 3, 88, 1077, 523, 128, 12, 43, 9029, 28, 5, 1240, 226, 10, 150, 682, 6, 8, 589, 19, 3, 23, 163, 43, 1911, 1135, 7, 11, 62, 26, 1496, 1135, 7, 326, 233, 836, 6961, 10, 3, 32, 107, 82, 6, 24, 31, 7, 626, 55, 3, 88, 31, 7, 882, 3, 9, 3164, 4024, 6, 68, 3, 88, 243, 62, 26, 1496, 1135, 2993, 1399, 5, 1240, 226, 10, 248, 258, 55, 2049, 21, 48, 55, 3, 10, 61, 836, 6961, 10, 4244, 1024, 1024, 6, 150, 682, 55, 78, 6, 54, 3, 23, 1499, 376, 39, 381, 58, 1240, 226, 10, 13, 503, 3, 10, 61, 836, 6961, 10, 3, 88, 31, 

In [15]:
# input ids => dialogue -> token ids

# 1 => EOS, 0 => padding

# attention mask
# labels - target => summary token

In [16]:
len(train_dataset[0]["input_ids"])

512

In [17]:
type(train_dataset)

list

## Working with our Model

In [18]:
# NLP => genration task

model = T5ForConditionalGeneration.from_pretrained("t5-small")

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  242MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [19]:
import torch

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("device : ", device)
model.to(device)

device :  cuda


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [20]:
# Training Arguments

training_args = TrainingArguments(
    output_dir = "./results",

    num_train_epochs=6,
    weight_decay=0.01,

    per_device_train_batch_size= 8,
    per_device_eval_batch_size= 8,

    eval_strategy="epoch",
    save_strategy="epoch",

    warmup_steps= 500
    # 0 => lr default
)

In [21]:
trainer = Trainer(
    model= model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

In [22]:
# train the model
trainer.train()

Epoch,Training Loss,Validation Loss
1,3.789052,0.113816
2,0.120513,0.105365
3,0.112496,0.103283
4,0.108828,0.102013
5,0.106932,0.101379
6,0.105691,0.101189


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3000, training_loss=0.7239188092549642, metrics={'train_runtime': 2390.1025, 'train_samples_per_second': 10.041, 'train_steps_per_second': 1.255, 'total_flos': 3248203235328000.0, 'train_loss': 0.7239188092549642, 'epoch': 6.0})

In [ ]:
# model load => fine-tune => saave the model

In [51]:
model.save_pretrained("./saved_summary_model")
tokenizer.save_pretrained("./saved_summary_model")

Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.32s/it]


('./saved_summary_model\\tokenizer_config.json',
 './saved_summary_model\\tokenizer.json')

In [53]:
model = T5ForConditionalGeneration.from_pretrained("./saved_summary_model")
tokenizer = T5Tokenizer.from_pretrained("./saved_summary_model")

Loading weights: 100%|██████████| 131/131 [00:00<00:00, 431.33it/s]


## Test the core logic for summarizaation

In [56]:
def summarize_dialogue(dialogue):
    dialogue = clean_data(dialogue) # clean

    # tokenize
    inputs = tokenizer(
        dialogue,
        padding="max_length",
        max_length=512,
        truncation=True,
        return_tensors="pt"
    )

    # generate the summary => token ids
    targets = model.generate(
        input_ids = inputs["input_ids"],
        attention_mask= inputs["attention_mask"],
        max_length=150,
        num_beams=4,
        early_stopping=True
    )

    # decoded our output
    summary = tokenizer.decode(targets[0], skip_special_tokens=True)  # EOS, SEP
    return summary

In [57]:
test_dialogue = """ 
Reporter: In today's technology news, artificial intelligence continues to expand rapidly across industries, from healthcare to finance and education. Recent reports suggest that AI adoption has significantly increased over the past few years.

Reporter: Companies are investing heavily in machine learning systems to automate tasks, improve decision-making, and enhance customer experiences. However, this growth has also raised questions about job displacement and ethical concerns.

Expert: AI systems are becoming more capable due to advances in deep learning and access to large datasets. These models can now perform complex tasks such as language understanding, image recognition, and even code generation.

Expert: At the same time, there are valid concerns about bias in AI models, as they often reflect the data they are trained on. Ensuring fairness and transparency is becoming a key area of research.

Reporter: Governments and organizations are beginning to introduce regulations to guide the development and deployment of AI technologies. The goal is to balance innovation with accountability.

Expert: Another challenge is explainability. Many modern AI systems, especially deep neural networks, operate as “black boxes,” making it difficult to understand how decisions are made.

Reporter: Experts also highlight the importance of responsible AI development, including data privacy, security, and long-term societal impact.

Expert: Looking ahead, collaboration between researchers, policymakers, and industry leaders will be crucial to ensure that AI systems are developed and used in a safe and beneficial way.
"""

summary = summarize_dialogue(test_dialogue)

print("Summary: ", summary)

Summary:  experts highlight the importance of responsible ai development, including data privacy, security, and long-term societal impact. ensuring fairness and transparency is becoming a key area of research.
